In [1]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

from qiskit.quantum_info import SparsePauliOp, Operator
from qiskit.primitives import StatevectorEstimator

In [2]:
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke     # 127-qubit Eagle processor
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator

In [3]:
def create_universal_ansatz(num_qubits, num_layers):
    theta = ParameterVector("θ", num_qubits * 3 * num_layers + num_qubits * 2 * (num_layers - 1))
    ansatz = QuantumCircuit(num_qubits)
    
    param_idx = 0
    for layer in range(num_layers):
        # Single-qubit rotations: Ry, Rz on each qubit
        for q in range(num_qubits):
            ansatz.ry(theta[param_idx], q)
            param_idx += 1
            ansatz.rz(theta[param_idx], q)
            param_idx += 1
        
        # Entangling layer (CX ladder)
        if layer < num_layers - 1:
            for q in range(num_qubits - 1):
                ansatz.cx(q, q + 1)
            # Optional: wrap around
            if num_qubits > 2:
                ansatz.cx(num_qubits - 1, 0)
    
    # Final single-qubit rotations
    for q in range(num_qubits):
        ansatz.ry(theta[param_idx], q)
        param_idx += 1
        ansatz.rz(theta[param_idx], q)
        param_idx += 1
    
    return ansatz, theta

# What follows is a usage example
ansatz, theta = create_universal_ansatz(2, num_layers=2)
print(ansatz.draw())

     ┌──────────┐┌──────────┐     ┌──────────┐┌──────────┐ ┌──────────┐»
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■──┤ Ry(θ[4]) ├┤ Rz(θ[5]) ├─┤ Ry(θ[8]) ├»
     ├──────────┤├──────────┤┌─┴─┐├──────────┤├──────────┤┌┴──────────┤»
q_1: ┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├┤ Ry(θ[6]) ├┤ Rz(θ[7]) ├┤ Ry(θ[10]) ├»
     └──────────┘└──────────┘└───┘└──────────┘└──────────┘└───────────┘»
«      ┌──────────┐
«q_0: ─┤ Rz(θ[9]) ├
«     ┌┴──────────┤
«q_1: ┤ Rz(θ[11]) ├
«     └───────────┘


In [4]:
def hardware_efficient_ansatz(num_qubits, num_layers):
    theta = ParameterVector("θ", num_qubits * 2 * num_layers)
    ansatz = QuantumCircuit(num_qubits)
    
    param_idx = 0
    for layer in range(num_layers):
        for q in range(num_qubits):
            ansatz.ry(theta[param_idx], q)
            param_idx += 1
            ansatz.rz(theta[param_idx], q)
            param_idx += 1
        for q in range(num_qubits - 1):
            ansatz.cx(q, q + 1)
    
    return ansatz, theta

# Usage example
ansatz, theta = hardware_efficient_ansatz(2, num_layers=2)
print(ansatz.draw())

     ┌──────────┐┌──────────┐     ┌──────────┐┌──────────┐     
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■──┤ Ry(θ[4]) ├┤ Rz(θ[5]) ├──■──
     ├──────────┤├──────────┤┌─┴─┐├──────────┤├──────────┤┌─┴─┐
q_1: ┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├┤ Ry(θ[6]) ├┤ Rz(θ[7]) ├┤ X ├
     └──────────┘└──────────┘└───┘└──────────┘└──────────┘└───┘


# First just the simple example, no obfuscation 

## Define the Hamiltonian

In [5]:
# Simple H = X x Y
H = SparsePauliOp("XY")   

## Print tyhe exact spectrum

In [6]:
# Print exact spectrum
eigvals = np.linalg.eigvalsh(Operator(H).data)
print(f"Exact eigenvalues: {np.round(eigvals, 6)}")
print(f"Exact ground state energy: {eigvals[0]:.6f}\n")

Exact eigenvalues: [-1. -1.  1.  1.]
Exact ground state energy: -1.000000



# Hardware efficient ansatz for 2 qubits

In [7]:
ansatz, theta = hardware_efficient_ansatz(2, num_layers=2)
print(ansatz.draw())

     ┌──────────┐┌──────────┐     ┌──────────┐┌──────────┐     
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■──┤ Ry(θ[4]) ├┤ Rz(θ[5]) ├──■──
     ├──────────┤├──────────┤┌─┴─┐├──────────┤├──────────┤┌─┴─┐
q_1: ┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├┤ Ry(θ[6]) ├┤ Rz(θ[7]) ├┤ X ├
     └──────────┘└──────────┘└───┘└──────────┘└──────────┘└───┘


In [8]:
# Build a noise model from a fake IBM backend
fake_backend = FakeSherbrooke()
noise_model = NoiseModel.from_backend(fake_backend)

# Transpile the parameterized ansatz to the backend's basis gates.
# This is required so the noise model is applied correctly.
#pass_manager = generate_preset_pass_manager(optimization_level=2, backend=fake_backend)
noisy_sim = AerSimulator(noise_model=noise_model)
pass_manager = generate_preset_pass_manager(optimization_level=2, backend=noisy_sim)

isa_ansatz = pass_manager.run(ansatz)

In [9]:
# Cost function

In [10]:
estimator = AerEstimator(
    options=dict(default_precision=0.05,
                 backend_options=dict(noise_model=noise_model))
)

def cost_func(params):
    job = estimator.run([(isa_ansatz, H, [params])])
    return float(job.result()[0].data.evs[0])

In [11]:
# Optimize
np.random.seed(42)
best_result = None

In [12]:
for trial in range(3):
    x0 = np.random.uniform(0, 2 * np.pi, size=len(theta))
    res = minimize(cost_func, x0, method="cobyla", options={"maxiter": 300})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

In [13]:
print(f"Optimized ground state energy : {best_result.fun:.6f}")
print(f"Optimal parameters θ          : {np.round(best_result.x, 4)}")
print(f"Optimization success          : {best_result.success}")

Optimized ground state energy : -1.072597
Optimal parameters θ          : [4.2212 7.0597 4.5161 3.4905 0.825  0.9813 1.4807 5.4744]
Optimization success          : True
